In [ ]:
!pip install qdrant-client sentence-transformers transformers torch

# 메타데이터 필터링 1

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client import models
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
# Qdrant 클라이언트 초기화 (API 키 노출 상태 유지)
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


In [ ]:
# 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# 상수
SIMILARITY_THRESHOLD = 0.7
COLLECTION_NAME = "son5"


In [ ]:
def search_disease_info(query_text, 질병_카테고리=None, 질병=None, 의도=None, limit=3):
    try:
        query_vector = encoder.encode(query_text).tolist()

        filter_conditions = []
        if 질병_카테고리:
            filter_conditions.append(
                models.FieldCondition(key="질병_카테고리", match=models.MatchValue(value=질병_카테고리))
            )
        if 질병:
            filter_conditions.append(
                models.FieldCondition(key="질병", match=models.MatchValue(value=질병))
            )
        if 의도:
            filter_conditions.append(
                models.FieldCondition(key="의도", match=models.MatchValue(value=의도))
            )

        search_params = models.SearchParams(hnsw_ef=128, exact=False)

        search_results = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,
            query_filter=models.Filter(must=filter_conditions) if filter_conditions else None,
            limit=limit,
            search_params=search_params
        )

        return [{"질병_카테고리": hit.payload.get("질병_카테고리", "N/A"),
                 "질병": hit.payload.get("질병", "N/A"),
                 "의도": hit.payload.get("의도", "N/A"),
                 "유사도": hit.payload.get("유사도", "N/A"),
                 "답변": hit.payload.get("답변", "N/A"),
                 "검색_유사도_점수": hit.score} for hit in search_results]
    except Exception as e:
        print(f"검색 중 오류 발생: {str(e)}")
        return []


In [ ]:
def generate_response(query, context=None):
    try:
        if context:
            input_text = f"질문: {query}\n컨텍스트: {context}\n답변:"
        else:
            input_text = f"질문: {query}\n답변:"

        input_ids = tokenizer.encode(input_text, return_tensors="pt")

        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_length=256,
                num_return_sequences=1,
                no_repeat_ngram_size=2,
                top_k=50,
                top_p=0.92,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        return tokenizer.decode(output[0], skip_special_tokens=True).strip()
    except Exception as e:
        print(f"응답 생성 중 오류 발생: {str(e)}")
        return "죄송합니다. 응답 생성 중 오류가 발생했습니다."


In [ ]:
def get_user_input(prompt, allow_empty=True):
    while True:
        user_input = input(prompt).strip()
        if allow_empty or user_input:
            return user_input if user_input else None
        print("입력이 필요합니다. 다시 시도해 주세요.")

In [ ]:
def advanced_rag_search():
    while True:
        query = get_user_input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ", allow_empty=False)
        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        질병_카테고리 = get_user_input("질병 카테고리를 입력하세요 (선택사항, 없으면 Enter): ")
        질병 = get_user_input("질병을 입력하세요 (선택사항, 없으면 Enter): ")
        의도 = get_user_input("의도를 입력하세요 (선택사항, 없으면 Enter): ")

        results = search_disease_info(query, 질병_카테고리, 질병, 의도)
        if results and results[0]['검색_유사도_점수'] >= SIMILARITY_THRESHOLD:
            best_match = results[0]
            print(f"\n'{query}'에 대한 검색 결과:\n")
            print(f"질병 카테고리: {best_match['질병_카테고리']}")
            print(f"질병: {best_match['질병']}")
            print(f"의도: {best_match['의도']}")
            print(f"DB 내 유사도: {best_match['유사도']}")
            print(f"검색 유사도 점수: {best_match['검색_유사도_점수']:.4f}")
            print("\nDB 참고 생성된 답변:")
            generated_response = generate_response(query, best_match['답변'])
            print(generated_response)
        else:
            print("\nDB를 참고하지 않고 생성된 답변:")
            generated_response = generate_response(query)
            print(generated_response)
        print("-" * 50)

if __name__ == "__main__":
    advanced_rag_search()

질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나
질병 카테고리를 입력하세요 (선택사항, 없으면 Enter): 
질병을 입력하세요 (선택사항, 없으면 Enter): 
의도를 입력하세요 (선택사항, 없으면 Enter): 증상

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

질병 카테고리: 근골격질환
질병: 아킬레스 건염
의도: 증상
DB 내 유사도: 0.8423691987991333
검색 유사도 점수: 0.8145

DB 참고 생성된 답변:
질문: 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나
컨텍스트: 아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로, 가능한 빠른 시일 내에 증상을 인지하고 의사의 도움을 받는 것이 중요합니다.
답변:네, 안녕하세요. 조수현입니다.운동 전후에는 적절한 스트레칭과 온찜질을 하고, 통증이 악화되지 않도록 주의하는 것이 좋습니다. 특히, 통증이 심하거나 운동 후 회복이 어려울 경우 즉시 전문의의 도움을 받아야 합니다.
 Q:운동 후 통증이 있는 경우 어떻게 해야 하나요? A:운동을 중단하고 휴식을 취하는 것이 가장 중요하며, 통증이 완화
---------------------------------

결과 :
 - 사용자가 직접 메타데이터를 입력하는 방식이 불편함
 - 그러다보니 입력 이벤트를 여러번 발생 시켜야함 -> 사용자가 귀찮아하거나 불편할 수 있음

# 메타데이터 필터링 2
- 메타데이터 직접 입력하는 방식 수정

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client import models
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

In [ ]:
# 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
# 상수
SIMILARITY_THRESHOLD = 0.7
COLLECTION_NAME = "son5"

In [ ]:
def get_unique_metadata():
    try:
        # Qdrant의 고유한 메타데이터 값을 가져옵니다.
        # client.scroll() 메서드를 사용하여 지정된 컬렉션에서 데이터를 가져옵니다.
        response = client.scroll(
            collection_name=COLLECTION_NAME,  # 데이터를 가져올 컬렉션의 이름
            scroll_filter=None,  # 필터를 적용하지 않고 모든 데이터를 가져옵니다.
            limit=10000,  # 한 번의 요청에서 가져올 최대 데이터 수를 설정합니다.
            with_payload=True,  # 데이터의 페이로드를 포함하여 가져옵니다.
            with_vectors=False  # 벡터 데이터는 포함하지 않도록 설정합니다.
        )

        # 고유한 메타데이터 값을 저장하기 위한 집합(set) 초기화
        categories = set()  # 질병 카테고리 저장용
        diseases = set()    # 질병 저장용
        intents = set()     # 의도 저장용

        # 응답에서 각 포인트를 반복하여 메타데이터를 추출합니다.
        for point in response[0]:
            # 포인트의 페이로드에서 "질병_카테고리", "질병", "의도" 값을 가져와 집합에 추가합니다.
            categories.add(point.payload.get("질병_카테고리", ""))  # 기본값은 빈 문자열
            diseases.add(point.payload.get("질병", ""))             # 기본값은 빈 문자열
            intents.add(point.payload.get("의도", ""))              # 기본값은 빈 문자열

        # 집합에서 빈 문자열을 제거합니다.
        categories.discard("")  # 빈 문자열 제거
        diseases.discard("")    # 빈 문자열 제거
        intents.discard("")     # 빈 문자열 제거

        # 고유한 메타데이터 값을 리스트로 변환하여 반환합니다.
        return list(categories), list(diseases), list(intents)
    except Exception as e:
        # 오류 발생 시 오류 메시지를 출력하고 빈 리스트를 반환합니다.
        print(f"메타데이터 가져오기 중 오류 발생: {str(e)}")
        return [], [], []  # 빈 리스트를 반환하여 호출자에게 오류를 알립니다.


In [ ]:
# 메타데이터 초기화
# get_unique_metadata() 함수를 호출하여 질병 카테고리, 질병, 의도 정보를 가져와 각각의 변수에 할당합니다.
DISEASE_CATEGORIES, DISEASES, INTENTS = get_unique_metadata()

def extract_metadata(query, categories, diseases, intents):
    # 입력된 쿼리를 소문자로 변환하여 대소문자 구분 없이 검색할 수 있도록 합니다.
    query_lower = query.lower()

    # 쿼리에서 첫 번째로 발견되는 질병 카테고리를 찾습니다.
    # categories 리스트를 순회하며, 쿼리와 일치하는 카테고리를 찾아 반환하고, 없으면 None을 반환합니다.
    category = next((cat for cat in categories if cat in query_lower), None)

    # 쿼리에서 첫 번째로 발견되는 질병을 찾습니다.
    # diseases 리스트를 순회하며, 쿼리와 일치하는 질병을 찾아 반환하고, 없으면 None을 반환합니다.
    disease = next((dis for dis in diseases if dis in query_lower), None)

    # 쿼리에서 첫 번째로 발견되는 의도를 찾습니다.
    # intents 리스트를 순회하며, 쿼리와 일치하는 의도를 찾아 반환하고, 없으면 None을 반환합니다.
    intent = next((intent for intent in intents if intent in query_lower), None)

    # 찾은 카테고리, 질병, 의도를 반환합니다.
    return category, disease, intent


In [ ]:
def search_disease_info(query_text, limit=3):
    try:
        # 입력된 쿼리 텍스트를 인코딩하여 벡터 형태로 변환합니다.
        query_vector = encoder.encode(query_text).tolist()

        # 쿼리 텍스트에서 메타데이터를 추출합니다.
        category, disease, intent = extract_metadata(query_text, DISEASE_CATEGORIES, DISEASES, INTENTS)

        # 검색 필터 조건을 저장할 리스트 초기화
        filter_conditions = []
        # 카테고리가 존재할 경우 필터 조건에 추가
        if category:
            filter_conditions.append(
                models.FieldCondition(key="질병_카테고리", match=models.MatchValue(value=category))
            )
        # 질병이 존재할 경우 필터 조건에 추가
        if disease:
            filter_conditions.append(
                models.FieldCondition(key="질병", match=models.MatchValue(value=disease))
            )
        # 의도가 존재할 경우 필터 조건에 추가
        if intent:
            filter_conditions.append(
                models.FieldCondition(key="의도", match=models.MatchValue(value=intent))
            )

        # 검색 파라미터 설정 (HNSW 알고리즘의 ef 값과 정확도 설정)
        search_params = models.SearchParams(hnsw_ef=128, exact=False)

        # Qdrant에서 검색 요청을 보냅니다.
        search_results = client.search(
            collection_name=COLLECTION_NAME,  # 검색할 컬렉션의 이름
            query_vector=query_vector,         # 쿼리 벡터
            query_filter=models.Filter(must=filter_conditions) if filter_conditions else None,  # 필터 조건
            limit=limit,                       # 검색 결과의 최대 개수
            search_params=search_params        # 검색 파라미터
        )

        # 검색 결과를 포맷하여 반환합니다.
        return [{"질병_카테고리": hit.payload.get("질병_카테고리", "N/A"),
                 "질병": hit.payload.get("질병", "N/A"),
                 "의도": hit.payload.get("의도", "N/A"),
                 "유사도": hit.payload.get("유사도", "N/A"),
                 "답변": hit.payload.get("답변", "N/A"),
                 "검색_유사도_점수": hit.score} for hit in search_results]
    except Exception as e:
        # 오류 발생 시 오류 메시지를 출력하고 빈 리스트를 반환합니다.
        print(f"검색 중 오류 발생: {str(e)}")
        return []  # 빈 리스트를 반환하여 호출자에게 오류를 알립니다.


In [ ]:
def generate_response(query, context=None):
    try:
        # 주어진 쿼리와 컨텍스트를 기반으로 입력 텍스트를 생성합니다.
        if context:
            # 컨텍스트가 제공된 경우, 질문과 컨텍스트를 포함한 입력 텍스트 형식
            input_text = f"질문: {query}\n컨텍스트: {context}\n답변:"
        else:
            # 컨텍스트가 없는 경우, 질문만 포함한 입력 텍스트 형식
            input_text = f"질문: {query}\n답변:"

        # 입력 텍스트를 토크나이즈하여 텐서 형태로 변환합니다.
        input_ids = tokenizer.encode(input_text, return_tensors="pt")

        # 모델의 출력을 생성하기 위한 비확률적 모드 설정
        with torch.no_grad():
            # 모델을 사용하여 응답을 생성합니다.
            output = model.generate(
                input_ids,  # 입력 ID
                max_length=256,  # 생성할 응답의 최대 길이
                num_return_sequences=1,  # 반환할 응답의 개수
                no_repeat_ngram_size=2,  # 반복되는 n-그램을 방지하기 위한 설정
                top_k=50,  # 상위 k개의 단어 중에서 선택
                top_p=0.92,  # 누적 확률이 p 이하인 단어 중에서 선택
                temperature=0.7,  # 샘플링의 온도 설정 (창의성 조절)
                do_sample=True,  # 샘플링을 통해 응답 생성
                pad_token_id=tokenizer.eos_token_id  # 패딩 토큰 ID 설정
            )

        # 생성된 응답을 디코딩하고 불필요한 특수 토큰을 제거한 후 반환합니다.
        return tokenizer.decode(output[0], skip_special_tokens=True).strip()
    except Exception as e:
        # 오류 발생 시 오류 메시지를 출력하고 사용자에게 오류 메시지를 반환합니다.
        print(f"응답 생성 중 오류 발생: {str(e)}")
        return "죄송합니다. 응답 생성 중 오류가 발생했습니다."


In [ ]:
def qdrant_based_rag_search():
    # 무한 루프를 시작하여 사용자의 입력을 계속해서 받습니다.
    while True:
        # 사용자에게 질문을 입력 받습니다.
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ").strip()

        # 사용자가 '끝내기'라고 입력하면 루프를 종료합니다.
        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        # 입력된 질문을 바탕으로 메타데이터를 추출합니다.
        category, disease, intent = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)
        print(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}")

        # 질병 정보를 검색합니다.
        results = search_disease_info(query)

        # 검색 결과가 있으며, 첫 번째 결과의 유사도 점수가 임계값을 초과하는지 확인합니다.
        if results and results[0]['검색_유사도_점수'] >= SIMILARITY_THRESHOLD:
            best_match = results[0]  # 가장 유사한 결과를 선택합니다.
            print(f"\n'{query}'에 대한 검색 결과:\n")
            print(f"질병 카테고리: {best_match['질병_카테고리']}")
            print(f"질병: {best_match['질병']}")
            print(f"의도: {best_match['의도']}")
            print(f"DB 내 유사도: {best_match['유사도']}")
            print(f"검색 유사도 점수: {best_match['검색_유사도_점수']:.4f}")

            # DB를 참고하여 생성된 답변을 생성합니다.
            print("\nDB 참고 생성된 답변:")
            generated_response = generate_response(query, best_match['답변'])
            print(generated_response)
        else:
            # 검색 결과가 없거나 유사도 점수가 임계값에 미치지 못할 경우
            print("\nDB를 참고하지 않고 생성된 답변:")
            generated_response = generate_response(query)  # DB를 참고하지 않고 답변을 생성합니다.
            print(generated_response)

        # 각 질문과 응답 사이에 구분선을 출력합니다.
        print("-" * 50)

# 스크립트가 직접 실행될 때 qdrant_based_rag_search() 함수를 호출합니다.
if __name__ == "__main__":
    qdrant_based_rag_search()


질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나
추출된 메타데이터: 카테고리=None, 질병=None, 의도=운동

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

질병 카테고리: 근골격질환
질병: 아킬레스 건염
의도: 운동
DB 내 유사도: 0.828995943069458
검색 유사도 점수: 0.7923

DB 참고 생성된 답변:
질문: 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나
컨텍스트: 아킬레스 건염은 아킬레스 건 부위에 염증이 생기는 질환으로, 주로 스포츠 활동이나 잘못된 운동 방법으로 인해 발생합니다. 이로 인해 통증이 나타나며, 아킬레스 건염의 주요 증상은 다음과 같습니다. 1. 통증 초기에는 활동 중 통증이 심해지고, 일정 수준의 움직임에 의해서도 계속되는 경우가 있습니다.2. 부종 통증과 함께 발뒤꿈치 부위에 부종이 발생하여 신발 신기가 힘들어질 수 있습니다.3. 염증 아킬레스 건 부위에 염증이 생기면 운동 시 통증이 더욱 심해지고, 종아리 부분까지 염증이 퍼질 수 있습니다.4. 보행 문제 운동을 하지 않을 때에도 통증이 지속되는 경우도 있습니다. 아킬레스 건염은 적절한 예방과 치료가 필요한 상황입니다. 통증을 관리하기 위해 적절한 스트레칭과 운동 방법을 유지하고, 필요에 따라 적절한 휴식을 취하는 것이 좋습니다. 만약 통증이 심해지거나 지속된다면 의료 전문가의 상담을 받아야 합니다.
답변:네, 대부분의 경우 적절한 휴식과 스트레칭을 통해 회복이 가능합니다. 특히 초기에 적절한 진단과 치료가 중요합니다. 통증이 심한 경우에는 냉찜질이나 온찜질을 통해 염증을 줄이고, 부기와 통증을 완화할 수 있으며, 경우에 따라 물리치료와 재활 운동을 병행하는 것이 도움이 될 수 있습니다.
 관련 질병 : 아
--------------------------------------------------
질문을 입력하세요 (종료하려면 '끝내기' 입